## Transformer古诗生成实战

In [16]:
data = [
    ("静夜思", "床前明月光，疑似地上霜。举头望明月，低头思故乡。"),
    ("春晓", "春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。"),
    ("相思", "红豆生南国，春来发几枝。愿君多采撷，此物最相思。"),
    ("江雪", "千山鸟飞绝，万径人踪灭。孤舟蓑笠翁，独钓寒江雪。"),
    ("登鹤雀楼", "白日依山尽，黄河入海流。欲穷千里目，更上一层楼。"),
]

In [17]:
def preprocess(data):
    src_sentences = []
    trg_sentences = []
    for src, trg in data:
        src_token = list(src)
        trg_token = list(trg)
        src_sentences.append(src_token)
        trg_sentences.append(trg_token)
    return src_sentences, trg_sentences


src_sentences, trg_sentences = preprocess(data)
src_sentences, trg_sentences

([['静', '夜', '思'], ['春', '晓'], ['相', '思'], ['江', '雪'], ['登', '鹤', '雀', '楼']],
 [['床',
   '前',
   '明',
   '月',
   '光',
   '，',
   '疑',
   '似',
   '地',
   '上',
   '霜',
   '。',
   '举',
   '头',
   '望',
   '明',
   '月',
   '，',
   '低',
   '头',
   '思',
   '故',
   '乡',
   '。'],
  ['春',
   '眠',
   '不',
   '觉',
   '晓',
   '，',
   '处',
   '处',
   '闻',
   '啼',
   '鸟',
   '。',
   '夜',
   '来',
   '风',
   '雨',
   '声',
   '，',
   '花',
   '落',
   '知',
   '多',
   '少',
   '。'],
  ['红',
   '豆',
   '生',
   '南',
   '国',
   '，',
   '春',
   '来',
   '发',
   '几',
   '枝',
   '。',
   '愿',
   '君',
   '多',
   '采',
   '撷',
   '，',
   '此',
   '物',
   '最',
   '相',
   '思',
   '。'],
  ['千',
   '山',
   '鸟',
   '飞',
   '绝',
   '，',
   '万',
   '径',
   '人',
   '踪',
   '灭',
   '。',
   '孤',
   '舟',
   '蓑',
   '笠',
   '翁',
   '，',
   '独',
   '钓',
   '寒',
   '江',
   '雪',
   '。'],
  ['白',
   '日',
   '依',
   '山',
   '尽',
   '，',
   '黄',
   '河',
   '入',
   '海',
   '流',
   '。',
   '欲',
   '穷',
   '千',
   '里',
   '目',
   '，',
   '更'

In [18]:
from collections import Counter

special_token = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]


def build_vocab(sentences):
    counter = Counter()

    for sentence in sentences:
        for token in sentence:
            counter[token] += 1

    vocab = special_token.copy()
    for word, count in counter.items():
        if word not in special_token:
            vocab.append(word)
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    return vocab, word2idx


src_vocab, src_word2idx = build_vocab(src_sentences)
trg_vocab, trg_word2idx = build_vocab(trg_sentences)

print(src_word2idx)
print(trg_word2idx)

{'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, '<UNK>': 3, '静': 4, '夜': 5, '思': 6, '春': 7, '晓': 8, '相': 9, '江': 10, '雪': 11, '登': 12, '鹤': 13, '雀': 14, '楼': 15}
{'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, '<UNK>': 3, '床': 4, '前': 5, '明': 6, '月': 7, '光': 8, '，': 9, '疑': 10, '似': 11, '地': 12, '上': 13, '霜': 14, '。': 15, '举': 16, '头': 17, '望': 18, '低': 19, '思': 20, '故': 21, '乡': 22, '春': 23, '眠': 24, '不': 25, '觉': 26, '晓': 27, '处': 28, '闻': 29, '啼': 30, '鸟': 31, '夜': 32, '来': 33, '风': 34, '雨': 35, '声': 36, '花': 37, '落': 38, '知': 39, '多': 40, '少': 41, '红': 42, '豆': 43, '生': 44, '南': 45, '国': 46, '发': 47, '几': 48, '枝': 49, '愿': 50, '君': 51, '采': 52, '撷': 53, '此': 54, '物': 55, '最': 56, '相': 57, '千': 58, '山': 59, '飞': 60, '绝': 61, '万': 62, '径': 63, '人': 64, '踪': 65, '灭': 66, '孤': 67, '舟': 68, '蓑': 69, '笠': 70, '翁': 71, '独': 72, '钓': 73, '寒': 74, '江': 75, '雪': 76, '白': 77, '日': 78, '依': 79, '尽': 80, '黄': 81, '河': 82, '入': 83, '海': 84, '流': 85, '欲': 86, '穷': 87, '里': 88, '目': 89, '更': 90, '一': 91, '层': 92, '楼': 

In [19]:
import torch


def tokenize(sentence, word2idx):
    tokens = [word2idx.get(token, word2idx["<UNK>"]) for token in sentence]
    return tokens


process_data_src = []
process_data_trg = []
for src, trg in zip(src_sentences, trg_sentences):
    src_numerical = tokenize(src, src_word2idx)
    trg_numerical = [trg_word2idx["<BOS>"]] + tokenize(trg, trg_word2idx) + [trg_word2idx["<EOS>"]]
    process_data_trg.append(torch.LongTensor(trg_numerical))
    process_data_src.append(torch.LongTensor(src_numerical))

process_data_trg

[tensor([ 1,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,  6,  7,
          9, 19, 17, 20, 21, 22, 15,  2]),
 tensor([ 1, 23, 24, 25, 26, 27,  9, 28, 28, 29, 30, 31, 15, 32, 33, 34, 35, 36,
          9, 37, 38, 39, 40, 41, 15,  2]),
 tensor([ 1, 42, 43, 44, 45, 46,  9, 23, 33, 47, 48, 49, 15, 50, 51, 40, 52, 53,
          9, 54, 55, 56, 57, 20, 15,  2]),
 tensor([ 1, 58, 59, 31, 60, 61,  9, 62, 63, 64, 65, 66, 15, 67, 68, 69, 70, 71,
          9, 72, 73, 74, 75, 76, 15,  2]),
 tensor([ 1, 77, 78, 79, 59, 80,  9, 81, 82, 83, 84, 85, 15, 86, 87, 58, 88, 89,
          9, 90, 13, 91, 92, 93, 15,  2])]

In [20]:
import torch.nn.utils.rnn as rnn

processed_data_src_pad = rnn.pad_sequence(process_data_src, batch_first=True, padding_value=src_word2idx["<PAD>"])
processed_data_trg_pad = rnn.pad_sequence(process_data_trg, batch_first=True, padding_value=trg_word2idx["<PAD>"])

processed_data_src_pad, processed_data_trg_pad

(tensor([[ 4,  5,  6,  0],
         [ 7,  8,  0,  0],
         [ 9,  6,  0,  0],
         [10, 11,  0,  0],
         [12, 13, 14, 15]]),
 tensor([[ 1,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,  6,  7,
           9, 19, 17, 20, 21, 22, 15,  2],
         [ 1, 23, 24, 25, 26, 27,  9, 28, 28, 29, 30, 31, 15, 32, 33, 34, 35, 36,
           9, 37, 38, 39, 40, 41, 15,  2],
         [ 1, 42, 43, 44, 45, 46,  9, 23, 33, 47, 48, 49, 15, 50, 51, 40, 52, 53,
           9, 54, 55, 56, 57, 20, 15,  2],
         [ 1, 58, 59, 31, 60, 61,  9, 62, 63, 64, 65, 66, 15, 67, 68, 69, 70, 71,
           9, 72, 73, 74, 75, 76, 15,  2],
         [ 1, 77, 78, 79, 59, 80,  9, 81, 82, 83, 84, 85, 15, 86, 87, 58, 88, 89,
           9, 90, 13, 91, 92, 93, 15,  2]]))

In [26]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(processed_data_src_pad, processed_data_trg_pad)
dataloader =DataLoader(dataset, batch_size=1, shuffle=False)

for src_seq, trg_seq in dataloader:
    print(src_seq)
    print(trg_seq)
    break

tensor([[4, 5, 6, 0]])
tensor([[ 1,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,  6,  7,
          9, 19, 17, 20, 21, 22, 15,  2]])


In [27]:
import math
from torch import nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=128):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

In [32]:

class PoemGenerator(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers, n_decoder_layers):
        super().__init__()
        self.encoder_embedding = nn.Embedding(len(src_vocab), d_model)
        self.decoder_embedding = nn.Embedding(len(trg_vocab), d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(d_model=d_model, dim_feedforward=d_ff, nhead=n_heads,
                                          num_decoder_layers=n_decoder_layers, num_encoder_layers=n_encoder_layers,
                                          batch_first=True)
        self.fc = nn.Linear(d_model, len(trg_vocab))

    def forward(self, src, trg):
        batch_size, en_seq_len = trg.shape
        mask = nn.Transformer.generate_square_subsequent_mask(en_seq_len)

        encoder_inputs = self.pos_encoder(self.encoder_embedding(src))
        decoder_inputs = self.pos_encoder(self.decoder_embedding(trg))

        outputs = self.transformer(encoder_inputs, decoder_inputs, tgt_mask=mask)

        return self.fc(outputs)


In [38]:
import torch.optim as optim

model = PoemGenerator(d_model=256, d_ff=2048, n_heads=8, n_encoder_layers=4, n_decoder_layers=4)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(ignore_index=trg_word2idx['<PAD>'])


In [39]:
epochs = 100

for step in range(epochs):
    for src, trg in dataloader:
        decoder_input = trg[:, :-1]
        decoder_target = trg[:, 1:]

        mask = nn.Transformer.generate_square_subsequent_mask(decoder_input.size(1))
        decoder_outputs = model(src, decoder_input)

        loss = criterion(decoder_outputs.view(-1, decoder_outputs.size(-1)), decoder_target.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'epoch:{step + 1},loss:{loss.item()}')

epoch:1,loss:4.741613388061523
epoch:2,loss:4.592840194702148
epoch:3,loss:4.382561206817627
epoch:4,loss:4.148886203765869
epoch:5,loss:3.8100669384002686
epoch:6,loss:3.686920404434204
epoch:7,loss:3.5193960666656494
epoch:8,loss:3.300001859664917
epoch:9,loss:2.9948906898498535
epoch:10,loss:2.8451015949249268
epoch:11,loss:2.6017751693725586
epoch:12,loss:2.3155832290649414
epoch:13,loss:2.132641077041626
epoch:14,loss:1.8100836277008057
epoch:15,loss:1.885258674621582
epoch:16,loss:1.506630539894104
epoch:17,loss:1.347380518913269
epoch:18,loss:1.1144115924835205
epoch:19,loss:0.9903718829154968
epoch:20,loss:0.8286465406417847
epoch:21,loss:0.8527459502220154
epoch:22,loss:0.6878305077552795
epoch:23,loss:0.7093874216079712
epoch:24,loss:0.5493320226669312
epoch:25,loss:0.519773542881012
epoch:26,loss:0.5173356533050537
epoch:27,loss:0.4179232716560364
epoch:28,loss:0.40870752930641174
epoch:29,loss:0.38999512791633606
epoch:30,loss:0.37655109167099
epoch:31,loss:0.35620638728141

In [45]:
def generator(src_sentences,model):
    src_token = torch.LongTensor(tokenize(src_sentences, src_word2idx))
    src_embedding = model.encoder_embedding(src_token).unsqueeze(0)
    positionalEncoding = PositionalEncoding(256)
    src_embedding = positionalEncoding(src_embedding)

    encoder_outputs = model.transformer.encoder(src_embedding)
    decoder_inputs = [src_word2idx['<BOS>']]

    for _ in range(50):
        with torch.no_grad():
            decoder_inputs_tensor = torch.LongTensor(decoder_inputs)
            decoder_inputs_tensor = model.decoder_embedding(decoder_inputs_tensor).unsqueeze(0)
            positionalEncoding = PositionalEncoding(256)
            decoder_inputs_tensor = positionalEncoding(decoder_inputs_tensor)
            mask = nn.Transformer.generate_square_subsequent_mask(decoder_inputs_tensor.size(1))
            output = model.transformer.decoder(decoder_inputs_tensor, encoder_outputs, tgt_mask=mask)
            output = model.fc(output)
            pred_token = output[:, -1, :].argmax().item()
            decoder_inputs.append(pred_token)
            if pred_token == trg_word2idx['<EOS>']:
                break
    return ' '.join(trg_vocab[idx] for idx in decoder_inputs[1:-1])

test_sentence = '春晓'

print(generator(test_sentence, model))


春 眠 不 觉 晓 ， 处 处 闻 啼 鸟 。 夜 来 风 雨 声 ， 花 落 知 多 少 。
